In [14]:
!apt-get update -qq
!apt-get install -y pciutils zstd curl

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pciutils is already the newest version (1:3.7.0-6).
zstd is already the newest version (1.4.8+dfsg-3build1).
curl is already the newest version (7.81.0-1ubuntu1.25).
0 upgraded, 0 newly installed, 0 to remove and 169 not upgraded.


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [19]:
import subprocess
import time

server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Servidor iniciado.")

Servidor iniciado.


In [ ]:
!ollama pull qwen3:14b

### EXTRAÇÃO EXPLÍCITA ABAIXO


In [ ]:
import requests
import json

# ==========================
# Lê os arquivos
# ==========================

with open("PromptDeExtracaoDePerguntasExplicitasV4.json", "r", encoding="utf-8") as f:
    prompt = f.read()

with open("audio_reuniao.txt", "r", encoding="utf-8") as f:
    reuniao = f.read()

prompt_final = prompt + "\n\n" + reuniao

# ==========================
# Payload
# ==========================

payload = {
    "model": "qwen3:14b",
    "prompt": prompt_final,
    "stream": False,
    "think": True,
    "options": {
        "temperature": 0,
        "top_p": 1,
        "top_k": 1,
        "repeat_penalty": 1.0,
        "num_ctx": 32768,
        "num_predict": -1,
        "seed": 42
    }
}

print("Enviando requisição ao Ollama...")

# ==========================
# Inferência
# ==========================

response = requests.post(
    "http://localhost:11434/api/generate",
    json=payload,
    timeout=None
)

response.raise_for_status()

dados = response.json()

# ==========================
# Mostrar tudo que a API devolveu
# ==========================

print("\n===== CAMPOS RETORNADOS =====")
print(list(dados.keys()))

print("\n===== JSON COMPLETO =====")
print(json.dumps(dados, indent=2, ensure_ascii=False))

# ==========================
# Salva o JSON bruto
# ==========================

with open("RespostaCompletaAPI.json", "w", encoding="utf-8") as f:
    json.dump(dados, f, indent=2, ensure_ascii=False)

# ==========================
# Salva somente a resposta final
# ==========================

resultado = dados.get("response", "")

# Converte a resposta (texto) em objeto Python
resultado_json = json.loads(resultado)

# Salva como JSON formatado
with open("PerguntasExplicitas.json", "w", encoding="utf-8") as f:
    json.dump(resultado_json, f, indent=2, ensure_ascii=False)

# ==========================
# Caso exista thinking separado
# ==========================

if "thinking" in dados:
    with open("Thinking.txt", "w", encoding="utf-8", newline="\n") as f:
        f.write(dados["thinking"])

if "reasoning" in dados:
    with open("Reasoning.txt", "w", encoding="utf-8", newline="\n") as f:
        f.write(dados["reasoning"])

print("\nArquivos gerados:")
print(" - PerguntasExplicitas.txt")
print(" - RespostaCompletaAPI.json")

if "thinking" in dados:
    print(" - Thinking.txt")

if "reasoning" in dados:
    print(" - Reasoning.txt")

print("\nInferência concluída.")

Enviando requisição ao Ollama...


** ETAPAS DE EXTRAÇÃO IMPLÍCITA ABAIXO**

### SUMARIZAÇÃO

In [ ]:
import requests
import json

# ==========================
# Lê os arquivos
# ==========================

with open("PromptSumarizacaoV1.txt", "r", encoding="utf-8") as f:
    prompt = f.read()

with open("audio_reuniao.txt", "r", encoding="utf-8") as f:
    reuniao = f.read()

prompt_final = prompt + "\n\n" + reuniao

# ==========================
# Payload
# ==========================

payload = {
    "model": "qwen3:14b",
    "prompt": prompt_final,
    "stream": False,
    "think": True,
    "options": {
        "temperature": 0,
        "top_p": 1,
        "top_k": 1,
        "repeat_penalty": 1.0,
        "num_ctx": 32768,
        "num_predict": -1,
        "seed": 42
    }
}

print("Gerando sumarização...")

response = requests.post(
    "http://localhost:11434/api/generate",
    json=payload,
    timeout=None
)

response.raise_for_status()

dados = response.json()

with open("RespostaCompletaAPI_Sumarizacao.json", "w", encoding="utf-8") as f:
    json.dump(dados, f, indent=2, ensure_ascii=False)

sumarizacao = dados.get("response", "")

with open("Sumarizacao.txt", "w", encoding="utf-8", newline="\n") as f:
    f.write(sumarizacao)

print("Sumarização concluída.")

Gerando sumarização...
Sumarização concluída.


### GERAÇÃO IMPLÍCITA

In [ ]:
import requests
import json

# ==========================
# Lê os arquivos
# ==========================

with open("PromptGerador.txt", "r", encoding="utf-8") as f:
    prompt = f.read()

with open("Sumarizacao.txt", "r", encoding="utf-8") as f:
    sumarizacao = f.read()

with open("audio_reuniao.txt", "r", encoding="utf-8") as f:
    reuniao = f.read()

prompt_final = (
    prompt
    + "\n\n"
    + sumarizacao
    + "\n\n"
    + reuniao
)

# ==========================
# Payload
# ==========================

payload = {
    "model": "qwen3:14b",
    "prompt": prompt_final,
    "stream": False,
    "think": True,
    "options": {
        "temperature": 0,
        "top_p": 1,
        "top_k": 1,
        "repeat_penalty": 1.0,
        "num_ctx": 32768,
        "num_predict": -1,
        "seed": 42
    }
}

print("Extraindo perguntas...")

response = requests.post(
    "http://localhost:11434/api/generate",
    json=payload,
    timeout=None
)

response.raise_for_status()

dados = response.json()

with open("RespostaCompletaAPI_Extracao.json", "w", encoding="utf-8") as f:
    json.dump(dados, f, indent=2, ensure_ascii=False)

perguntas = dados.get("response", "")

with open("PerguntasImplicitas.txt", "w", encoding="utf-8", newline="\n") as f:
    f.write(perguntas)

print("Extração concluída.")

Extraindo perguntas...
Extração concluída.


REFINAMENTO IMPLÍCITO

In [ ]:
import requests
import json

# ==========================
# Lê os arquivos
# ==========================

with open("PromptRefinadorPerguntas.txt", "r", encoding="utf-8") as f:
    prompt = f.read()

with open("PerguntasImplicitas.txt", "r", encoding="utf-8") as f:
    perguntas = f.read()

with open("audio_reuniao.txt", "r", encoding="utf-8") as f:
    reuniao = f.read()

prompt_final = (
    prompt
    + "\n\n"
    + perguntas
    + "\n\n"
    + reuniao
)

# ==========================
# Payload
# ==========================

payload = {
    "model": "qwen3:14b",
    "prompt": prompt_final,
    "stream": False,
    "think": True,
    "options": {
        "temperature": 0,
        "top_p": 1,
        "top_k": 1,
        "repeat_penalty": 1.0,
        "num_ctx": 32768,
        "num_predict": -1,
        "seed": 42
    }
}

print("Refinando perguntas...")

response = requests.post(
    "http://localhost:11434/api/generate",
    json=payload,
    timeout=None
)

response.raise_for_status()

dados = response.json()

with open("RespostaCompletaAPI_Refinamento.json", "w", encoding="utf-8") as f:
    json.dump(dados, f, indent=2, ensure_ascii=False)

resultado = dados.get("response", "")

with open("PerguntasImplicitasRefinadas.txt", "w", encoding="utf-8", newline="\n") as f:
    f.write(resultado)

print("Refinamento concluído.")

Refinando perguntas...
Refinamento concluído.
